# 20.15 — Vector databases & ANN infra

Vector databases make nearest-neighbor search operational by trading exactness for indexed candidate generation, recall, latency, and memory.

Embeddings supply the vector geometry; system design supplies latency and memory budgets. This notebook implements exact cosine search, IVF-style approximate candidate generation, recall@10, scan volume, and freshness pitfalls in NumPy.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2017)


def percentile(values, q):
    return float(np.percentile(np.asarray(values, dtype=float), q))


def softmax(logits, temperature=1.0):
    scaled = np.asarray(logits, dtype=float) / temperature
    shifted = scaled - np.max(scaled, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)


def cross_entropy(target_probs, pred_probs):
    clipped = np.clip(pred_probs, 1e-9, 1.0)
    return float(-np.sum(target_probs * np.log(clipped)))


def expected_calibration_error(confidence, correct, bins=10):
    confidence = np.asarray(confidence, dtype=float)
    correct = np.asarray(correct, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = len(confidence)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence >= lo) & (confidence < hi)
        if hi == 1.0:
            mask = (confidence >= lo) & (confidence <= hi)
        if np.any(mask):
            gap = abs(float(np.mean(confidence[mask])) - float(np.mean(correct[mask])))
            ece += float(np.mean(mask)) * gap
    return ece


def make_f17_workload_ladder(seed=2017):
    local_rng = np.random.default_rng(seed)
    return [
        {
            "rung": "D1",
            "name": "tiny hand-check",
            "requests": 60,
            "precision": "int8",
            "scale": 1.8 / 127.0,
            "shape": (8,),
            "load": 1.0,
            "data": np.array([0.73, -0.40, 1.20, -1.70, 0.05, 0.88, -0.91, 1.55]),
        },
        {
            "rung": "D2",
            "name": "clean tensor workload",
            "requests": 400,
            "precision": "int8",
            "scale": 2.5 / 127.0,
            "shape": (64, 32),
            "load": 1.7,
            "data": local_rng.normal(0.0, 0.65, size=(64, 32)),
        },
        {
            "rung": "D3",
            "name": "outliers and bad calibration",
            "requests": 900,
            "precision": "int8",
            "scale": 1.0 / 127.0,
            "shape": (128, 48),
            "load": 2.4,
            "data": np.vstack([
                local_rng.normal(0.0, 0.45, size=(124, 48)),
                local_rng.normal(0.0, 2.2, size=(4, 48)),
            ]),
        },
        {
            "rung": "D4",
            "name": "digits-like classifier weights",
            "requests": 1600,
            "precision": "mixed int8/fp16",
            "scale": 3.0 / 127.0,
            "shape": (10, 64),
            "load": 3.5,
            "data": local_rng.normal(0.0, 0.85, size=(10, 64)),
        },
        {
            "rung": "D5",
            "name": "production-scale load simulation",
            "requests": 5000,
            "precision": "per-channel int8",
            "scale": 4.0 / 127.0,
            "shape": (256, 128),
            "load": 6.0,
            "data": local_rng.normal(0.0, 0.95, size=(256, 128)),
        },
    ]


def preview_ladder(ladder):
    for rung in ladder:
        data = rung.get("data")
        shape = getattr(data, "shape", rung.get("shape"))
        print(f"{rung['rung']} | {rung['name']} | shape={shape} | requests={rung['requests']} | precision={rung['precision']}")
        print("sample:", np.asarray(data).reshape(-1)[:5])


## The concept, built once (D1)

Use $\cos(q,x)=q^\top x/(\|q\|\|x\|)$ and $\text{recall@}k=r/k$. The lesson's unit-norm dot $0.80$ gives cosine $0.80$, and $7$ recovered out of $10$ gives recall@$10=0.70$.

In [ ]:

def normalize_rows(values):
    values = np.asarray(values, dtype=float)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.maximum(norms, 1e-9)


def cosine_scores(query, vectors):
    query = np.asarray(query, dtype=float)
    vectors = np.asarray(vectors, dtype=float)
    query_norm = np.linalg.norm(query)
    vector_norms = np.linalg.norm(vectors, axis=1)
    return vectors @ query / np.maximum(vector_norms * query_norm, 1e-9)


def ann_search(query, vectors, probes=3, k=10, centroids=None, assignments=None):
    vectors = np.asarray(vectors, dtype=float)
    if centroids is None or assignments is None:
        scores = cosine_scores(query, vectors)
        order = np.argsort(-scores)[:k]
        return order, scores[order], len(vectors)
    centroid_scores = cosine_scores(query, centroids)
    probe_lists = np.argsort(-centroid_scores)[:probes]
    candidate_mask = np.isin(assignments, probe_lists)
    candidate_ids = np.flatnonzero(candidate_mask)
    candidate_scores = cosine_scores(query, vectors[candidate_ids])
    local_order = np.argsort(-candidate_scores)[:k]
    return candidate_ids[local_order], candidate_scores[local_order], len(candidate_ids)

query = np.array([1.0, 0.0])
candidate = np.array([[0.8, 0.6]])
cosine = float(cosine_scores(query, candidate)[0])
recall_at_10 = 7 / 10
scanned_3 = 3 * 120
scanned_8 = 8 * 120
memory_mb = 1_000_000 * 128 * 4 / 1_000_000

print("cosine", cosine)
print("recall@10", recall_at_10)
print("scanned 3", scanned_3)
print("scanned 8", scanned_8)
print("memory MB", memory_mb)

assert round(cosine, 2) == 0.80
assert recall_at_10 == 0.70
assert scanned_3 == 360
assert scanned_8 == 960
assert memory_mb == 512


The evaluator builds a small IVF index by assigning vectors to centroids, probes the closest lists, and compares approximate neighbors with exact top-10 neighbors.

In [ ]:

def make_ann_ladder(seed=2017):
    local_rng = np.random.default_rng(seed + 15)
    specs = [
        ("D1", "five hand vectors", 5, 2, 2, 1, 1),
        ("D2", "10k clean clustered vectors", 10_000, 16, 20, 3, 120),
        ("D3", "unnormalized vectors and stale inserts", 12_000, 24, 24, 3, 120),
        ("D4", "mixture embedding trace", 16_000, 32, 32, 5, 160),
        ("D5", "production ANN load simulation", 25_000, 48, 48, 8, 240),
    ]
    ladder = []
    for rung, name, n, dim, lists, probes, list_size in specs:
        centroids = normalize_rows(local_rng.normal(0.0, 1.0, size=(lists, dim)))
        assignments = local_rng.integers(0, lists, size=n)
        vectors = centroids[assignments] + local_rng.normal(0.0, 0.18, size=(n, dim))
        if rung != "D3":
            vectors = normalize_rows(vectors)
        queries = normalize_rows(local_rng.normal(0.0, 1.0, size=(12, dim)))
        ladder.append({
            "rung": rung,
            "name": name,
            "vectors": vectors,
            "queries": queries,
            "centroids": centroids,
            "assignments": assignments,
            "probes": probes,
            "list_size": list_size,
            "requests": len(queries),
            "precision": "float32 embeddings",
            "load": n / 10_000,
            "data": vectors[: min(n, 80)],
        })
    ladder[0]["vectors"] = normalize_rows(np.array([[1.0, 0.0], [0.8, 0.6], [0.0, 1.0], [-1.0, 0.0], [0.4, 0.9]]))
    ladder[0]["queries"] = np.array([[1.0, 0.0]])
    ladder[0]["centroids"] = normalize_rows(np.array([[1.0, 0.0], [0.0, 1.0]]))
    ladder[0]["assignments"] = np.array([0, 0, 1, 1, 1])
    return ladder


def evaluate_ann_workload(rung, probes=None, normalize=True):
    probes = rung["probes"] if probes is None else probes
    vectors = normalize_rows(rung["vectors"]) if normalize else np.asarray(rung["vectors"], dtype=float)
    centroids = normalize_rows(rung["centroids"])
    recalls = []
    scanned = []
    artifacts = []
    for query in rung["queries"]:
        exact_ids, exact_scores, exact_scanned = ann_search(query, vectors, probes=len(centroids), k=min(10, len(vectors)))
        ann_ids, ann_scores, ann_scanned = ann_search(query, vectors, probes=probes, k=min(10, len(vectors)), centroids=centroids, assignments=rung["assignments"])
        recovered = len(set(exact_ids).intersection(set(ann_ids)))
        recalls.append(recovered / len(exact_ids))
        scanned.append(ann_scanned)
        artifacts.append(ann_scores[:5])
    recall = float(np.mean(recalls))
    avg_scanned = float(np.mean(scanned))
    latency_ms = 1.0 + 0.002 * avg_scanned + 0.15 * probes
    return {
        "recall_at_10": recall,
        "avg_scanned": avg_scanned,
        "latency_ms": latency_ms,
        "artifact": np.concatenate(artifacts),
    }


## The dataset ladder

D1 has five hand vectors; D2 uses clean clusters; D3 adds unnormalized vectors and stale inserts; D4 is a mixture embedding trace; D5 is production-scale ANN load simulation.

In [ ]:

ladder = make_ann_ladder()
preview_ladder(ladder)


## Run the SAME method across D1–D5

In [ ]:

ann_results = []
for rung in ladder:
    result = evaluate_ann_workload(rung)
    result["rung"] = rung["rung"]
    result["name"] = rung["name"]
    ann_results.append(result)

print("rung | probes | recall@10 | avg_scanned | latency_ms")
for rung, result in zip(ladder, ann_results):
    print(f"{result['rung']} | {rung['probes']} | {result['recall_at_10']:.3f} | {result['avg_scanned']:.1f} | {result['latency_ms']:.2f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, result in zip(axes[0], ann_results):
    ax.plot(result["artifact"], marker="o", linewidth=1.0)
    ax.set_title(result["rung"] + " ANN scores")
    ax.set_ylim(-0.2, 1.05)

metric_curve = [result["recall_at_10"] for result in ann_results]
axes[1, 0].plot([rung["probes"] for rung in ladder], metric_curve, marker="o")
axes[1, 0].set_title("recall@10 curve")
axes[1, 0].set_xlabel("probes")
for ax, result in zip(axes[1, 1:], ann_results[1:]):
    ax.bar(["recall", "latency", "scanned"], [result["recall_at_10"], result["latency_ms"], result["avg_scanned"]])
    ax.set_title(result["rung"] + " operational panel")
plt.tight_layout()


## Pitfall on D5: reporting latency without recall

ANN can look fast by probing too little. Reproduce a low-probe miss rate, then choose probes under a recall constraint.

In [ ]:

d5 = ladder[-1]
fast = evaluate_ann_workload(d5, probes=1)
chosen = None
for probes in [1, 2, 4, 8, 12, 16, 24, 32, 48]:
    candidate = evaluate_ann_workload(d5, probes=probes)
    if candidate["recall_at_10"] >= 0.70:
        chosen = candidate
        chosen["probes"] = probes
        break

print("fast recall/latency", round(fast["recall_at_10"], 3), round(fast["latency_ms"], 2))
print("chosen probes", chosen["probes"])
print("chosen recall/latency", round(chosen["recall_at_10"], 3), round(chosen["latency_ms"], 2))
assert chosen["recall_at_10"] >= 0.70
assert chosen["recall_at_10"] > fast["recall_at_10"]


## Evaluate it + Practice

- Metric: track recall@10 with latency; compare with the no-skill baseline `exact search over all vectors`.
- Cheap sanity check: unit-norm dot 0.80 should equal cosine 0.80.
- Ablation: probe only one list.
- Failure signals: low latency paired with low recall or stale vectors.
- Production guardrail: monitor the metric by rung instead of averaging D1 with D5.

Practice 1: Change one D5 load or precision setting and predict the metric before running.

Practice 2: Add one operational constraint, such as memory budget or p95 latency budget, and mark the first rung that violates it.

Practice 3: Write a one-line launch rule that uses the metric and one pitfall guardrail.